In [ ]:
# Configure SCBFM_ROOT_DIR and optionally SCBFM_FIGURE_DIR before launching Jupyter.
from pathlib import Path
import os
import sys

_candidates = [Path(os.environ['SCBFM_REPO_DIR'])] if os.environ.get('SCBFM_REPO_DIR') else []
_candidates += [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in _candidates if (p / 'src' / 'main.py').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open Jupyter inside the checkout or set SCBFM_REPO_DIR.')
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from notebook_setup import ROOT_DIR, OUTPUT_DIR, FIGURE_DIR


In [ ]:
import re
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse

## Part I - Import and reorganization to homogene h5ad structure

In [ ]:
disignatlas = ad.read_h5ad(str(ROOT_DIR / 'datasets/DiSignAtlas/disease_annot_data.h5ad'))

In [ ]:
# replace gene symbols by ensembl IDs
disignatlas.var["symbol"] = disignatlas.var_names
disignatlas.var_names = disignatlas.var["ensg_id"].copy()

In [ ]:
disignatlas.write(str(ROOT_DIR / 'datasets/DiSignAtlas/disignatlas.h5ad'))

## Part II - statistics

In [ ]:
with open(str(REPO_ROOT / 'data/gene_list.txt')) as f:
    gene_list = [line.strip() for line in f if line.strip()]

In [ ]:
disignatlas = ad.read_h5ad(str(ROOT_DIR / 'datasets/DiSignAtlas/disignatlas.h5ad'))
print(disignatlas)

In [ ]:
gene_set = set(map(str, gene_list))
var_names = np.asarray(disignatlas.var_names.astype(str))

in_list_mask = np.array([g in gene_set for g in var_names], dtype=bool)
not_in_list_mask = ~in_list_mask
not_in_list_weights = not_in_list_mask.astype(np.float64)

chunk_size = 1000

sum_frac_nonzero = 0.0
sum_frac_reads = 0.0
n_obs_done = 0

for start in range(0, disignatlas.n_obs, chunk_size):
    end = min(start + chunk_size, disignatlas.n_obs)

    X_chunk = disignatlas.X[start:end]

    if sparse.issparse(X_chunk):
        X_chunk = X_chunk.tocsr()

        total_nonzero = np.asarray(X_chunk.getnnz(axis=1)).ravel()
        total_reads = np.asarray(X_chunk.sum(axis=1)).ravel()

        # Same as X_chunk[:, not_in_list_mask].sum(axis=1), but avoids sparse fancy indexing.
        not_in_list_reads = np.asarray(X_chunk @ not_in_list_weights).ravel()

        X_binary = X_chunk.copy()
        X_binary.data = np.ones_like(X_binary.data, dtype=np.float64)
        not_in_list_nonzero = np.asarray(X_binary @ not_in_list_weights).ravel()

        del X_binary

    else:
        X_chunk = np.asarray(X_chunk)

        total_nonzero = (X_chunk > 0).sum(axis=1)
        not_in_list_nonzero = (X_chunk[:, not_in_list_mask] > 0).sum(axis=1)

        total_reads = X_chunk.sum(axis=1)
        not_in_list_reads = X_chunk[:, not_in_list_mask].sum(axis=1)

    frac_nonzero_not_in_list = np.divide(
        not_in_list_nonzero,
        total_nonzero,
        out=np.zeros_like(total_nonzero, dtype=float),
        where=total_nonzero > 0,
    )

    frac_reads_not_in_list = np.divide(
        not_in_list_reads,
        total_reads,
        out=np.zeros_like(total_reads, dtype=float),
        where=total_reads > 0,
    )

    sum_frac_nonzero += frac_nonzero_not_in_list.sum()
    sum_frac_reads += frac_reads_not_in_list.sum()
    n_obs_done += end - start

    if start == 0 or n_obs_done % (10 * chunk_size) == 0 or end == disignatlas.n_obs:
        print(f"Processed {n_obs_done:,}/{disignatlas.n_obs:,} samples")

    del X_chunk

print("Average portion of non-zero genes NOT in gene_list:",
      sum_frac_nonzero / n_obs_done)

print("Average portion of total reads NOT in gene_list:",
      sum_frac_reads / n_obs_done)


## Part III - filter to gene list

In [ ]:
gene_list = [str(g) for g in gene_list]
gene_index = pd.Index(disignatlas.var_names.astype(str))

reorder_idx = gene_index.get_indexer(gene_list)
missing = [g for g, i in zip(gene_list, reorder_idx) if i < 0]
if missing:
    raise ValueError(f"{len(missing)} genes from gene_list are missing in disignatlas. First 20: {missing[:20]}")

out_path = str(ROOT_DIR / 'datasets/DiSignAtlas/disignatlas.h5ad')
chunk_size = 1000

chunks = []

for start in range(0, disignatlas.n_obs, chunk_size):
    end = min(start + chunk_size, disignatlas.n_obs)
    X_chunk = disignatlas.X[start:end, :]

    if sparse.issparse(X_chunk):
        # CSC handles column selection more reliably than CSR on some SciPy builds.
        X_chunk = X_chunk.tocsc()[:, reorder_idx].tocsr()
    else:
        X_chunk = np.asarray(X_chunk)[:, reorder_idx]

    chunk = ad.AnnData(
        X=X_chunk,
        obs=disignatlas.obs.iloc[start:end].copy(),
        var=pd.DataFrame(index=pd.Index(gene_list, name=disignatlas.var_names.name)),
    )
    chunks.append(chunk)

    print(f"Prepared {end:,}/{disignatlas.n_obs:,} samples")

disignatlas_aligned = ad.concat(chunks, axis=0, join="inner", merge="same")
disignatlas_aligned.var_names = pd.Index(gene_list, name=disignatlas.var_names.name)

disignatlas_aligned.write_h5ad(out_path)
print("Wrote:", out_path)
print("Shape:", disignatlas_aligned.shape)